# Scraper work in progress

In [15]:
import requests
import pandas as pd
from datetime import datetime
from ratelimit import limits, sleep_and_retry

@sleep_and_retry
@limits(calls=10, period=10)
def call_api(url):
	print(f"Getting {url}")
	response = requests.get(url)
	parsed = response.json()
	return parsed

In [2]:
case_summaries_json = call_api('https://api.oyez.org/cases?per_page=0')

cases = []

for case_summary in case_summaries_json:
    df = pd.json_normalize(case_summary)
    cases.append(df)
    
cases_df = pd.concat(cases)
case_summary_df = cases_df[['ID','name','href','docket_number','term']].set_index('ID')
case_summary_df.to_csv('case_summary.csv', index=True)

In [ ]:
case_summary_df = pd.read_csv('case_summary.csv')

print("Case table ready: ",datetime.now())

recent_cases = case_summary_df[case_summary_df['term'] >= "2000"]

oa_list = []

for index, case in recent_cases.iterrows():
    case_json = call_api(case['href'])
    case_id = case_json['ID']
    if case_json['oral_argument_audio'] is not None:
    	oa = pd.json_normalize(case_json['oral_argument_audio'])[['id','title','href']].set_index('id')
   		#oa['date'] = oa['title'].apply(lambda L:L.split(' - ')[1])
    	oa['case_id'] = case_id
    	oa = oa[['case_id','title','href']]
    	oa_list.append(oa)

oa_df = pd.concat(oa_list)
oa_df.to_csv('oa_summary.csv', index=True)

In [18]:
oa_summary_df = pd.read_csv('oa_summary.csv')
oa_summary_df = oa_summary_df[oa_summary_df['title'] != 'No oral argument']

oa_summary_df.head()

,id,case_id,title,href
1,23122,54856,"Oral Argument - November 08, 2000",https://api.oyez.org/case_media/oral_argument_...
2,21358,54857,"Oral Argument - February 26, 2001",https://api.oyez.org/case_media/oral_argument_...
3,21837,54858,"Oral Argument - October 30, 2000",https://api.oyez.org/case_media/oral_argument_...
4,22771,54859,"Oral Argument - November 08, 2000",https://api.oyez.org/case_media/oral_argument_...
5,21446,54860,"Oral Argument - October 10, 2000",https://api.oyez.org/case_media/oral_argument_...


In [111]:
petitioner_transcripts = []

for index, oa in oa_summary_df.iterrows():
    oa_json = call_api(oa['href'])
    oa_id = oa_json['id']
    transcript = oa_json['transcript']
    petitioner_transcript = transcript['sections'][0]
    for index, turn in enumerate(petitioner_transcript['turns']):
        if turn['speaker'] is not None:
            speaker = turn['speaker']['ID']
        else:
            speaker = respondent_transcript['turns'][index - 2]['speaker']['ID']
        for speech in turn['text_blocks']:
            text = speech['text']
            start = speech['start']
            stop = speech['stop']
            pt_dict = {"oa_id":oa_id,"speaker_id":speaker,"start":start,"stop":stop,"text":text}
            petitioner_transcripts.append(pt_dict)

petitioner_transcript_df = pd.DataFrame(petitioner_transcripts)
            
respondent_transcripts = []

for index, oa in oa_summary_df.iterrows():
    oa_json = call_api(oa['href'])
    oa_id = oa_json['id']
    transcript = oa_json['transcript']
    respondent_transcript = transcript['sections'][1]
    for index, turn in enumerate(respondent_transcript['turns']):
        if turn['speaker'] is not None:
            speaker = turn['speaker']['ID']
        else:
            speaker = respondent_transcript['turns'][index - 2]['speaker']['ID']
        for speech in turn['text_blocks']:
            text = speech['text']
            start = speech['start']
            stop = speech['stop']
            pt_dict = {"oa_id":oa_id,"speaker_id":speaker,"start":start,"stop":stop,"text":text}
            respondent_transcripts.append(pt_dict)
        
respondent_transcript_df = pd.DataFrame(respondent_transcripts)
        
rebuttal_transcripts = []

for index, oa in oa_summary_df.iterrows():
    oa_json = call_api(oa['href'])
    oa_id = oa_json['id']
    transcript = oa_json['transcript']
    rebuttal_transcript = transcript['sections'][2]
    for index, turn in enumerate(rebuttal_transcript['turns']):
        if turn['speaker'] is not None:
            speaker = turn['speaker']['ID']
        else:
            speaker = respondent_transcript['turns'][index - 2]['speaker']['ID']
        for speech in turn['text_blocks']:
            text = speech['text']
            start = speech['start']
            stop = speech['stop']
            pt_dict = {"oa_id":oa_id,"speaker_id":speaker,"start":start,"stop":stop,"text":text}
            rebuttal_transcripts.append(pt_dict)
        
rebuttal_transcript_df = pd.DataFrame(rebuttal_transcripts)

Getting https://api.oyez.org/case_media/oral_argument_audio/23122
Getting https://api.oyez.org/case_media/oral_argument_audio/21358
Getting https://api.oyez.org/case_media/oral_argument_audio/21837
Getting https://api.oyez.org/case_media/oral_argument_audio/22771
Getting https://api.oyez.org/case_media/oral_argument_audio/21446
Getting https://api.oyez.org/case_media/oral_argument_audio/21985
Getting https://api.oyez.org/case_media/oral_argument_audio/22240


IndexError: list index out of range

In [114]:
test_transcript = call_api('https://api.oyez.org/case_media/oral_argument_audio/22240')['transcript']
for index, turn in enumerate(test_transcript['sections'][0]['turns']):
    if turn['speaker']['ID'] is not None:
        print(index, turn['speaker']['ID'])
    else:
        print(index, 'N/A')

#for turn in test_transcript['sections'][0]['turns']:
#    print(turn['speaker']['ID'], turn['speaker']['name'])
    


Getting https://api.oyez.org/case_media/oral_argument_audio/22240
0 15141
1 38302
2 15042
3 38302
4 15042
5 38302
6 15042
7 38302
8 15042
9 38302
10 15042
11 38302


TypeError: 'NoneType' object is not subscriptable

In [88]:
petitioner_transcripts = []

oa_json = call_api('https://api.oyez.org/case_media/oral_argument_audio/23122')
oa_id = oa_json['id']
transcript = oa_json['transcript']
petitioner_transcript = transcript['sections'][0]
for index, turn in enumerate(petitioner_transcript['turns']):
    if turn['speaker'] is not None:
        speaker = turn['speaker']['ID']
    else:
        speaker = petitioner_transcript['turns'][index - 2]['speaker']['ID']
        #should be person before, this turn is an interruption
    for speech in turn['text_blocks']:
        text = speech['text']
        start = speech['start']
        stop = speech['stop']
        pt_dict = {"oa_id":oa_id,"speaker_id":speaker,"start":start,"stop":stop,"text":text}
        petitioner_transcripts.append(pt_dict)

respondent_transcripts = []

oa_json = call_api('https://api.oyez.org/case_media/oral_argument_audio/23122')
oa_id = oa_json['id']
transcript = oa_json['transcript']
respondent_transcript = transcript['sections'][1]
for index, turn in enumerate(respondent_transcript['turns']):
    if turn['speaker'] is not None:
        speaker = turn['speaker']['ID']
    else:
        speaker = respondent_transcript['turns'][index - 2]['speaker']['ID']
    for speech in turn['text_blocks']:
        text = speech['text']
        start = speech['start']
        stop = speech['stop']
        pt_dict = {"oa_id":oa_id,"speaker_id":speaker,"start":start,"stop":stop,"text":text}
        respondent_transcripts.append(pt_dict)
        
respondent_transcript_df = pd.DataFrame(respondent_transcripts)
        
rebuttal_transcripts = []

oa_json = call_api('https://api.oyez.org/case_media/oral_argument_audio/23122')
oa_id = oa_json['id']
transcript = oa_json['transcript']
rebuttal_transcript = transcript['sections'][2]
for index, turn in enumerate(rebuttal_transcript['turns']):
    if turn['speaker'] is not None:
        speaker = turn['speaker']['ID']
    else:
        speaker = rebuttal_transcript['turns'][index - 2]['speaker']['ID']
    for speech in turn['text_blocks']:
        text = speech['text']
        start = speech['start']
        stop = speech['stop']
        pt_dict = {"oa_id":oa_id,"speaker_id":speaker,"start":start,"stop":stop,"text":text}
        rebuttal_transcripts.append(pt_dict)
        
rebuttal_transcript_df = pd.DataFrame(rebuttal_transcripts)

Getting https://api.oyez.org/case_media/oral_argument_audio/23122
Getting https://api.oyez.org/case_media/oral_argument_audio/23122
Getting https://api.oyez.org/case_media/oral_argument_audio/23122


,oa_id,speaker_id,start,stop,text
0,23122,33864,3399.859,3400.887,Thank you.
1,23122,33864,3400.887,3402.146,I think I have two points.
2,23122,33864,3402.146,3412.477,"Justice Breyer, as we understand the issue tha..."
3,23122,33864,3412.477,3441.359,The Court quoted the same provision of the sta...
4,23122,33864,3441.359,3455.325,And what the Court said in Haggar was that the...
5,23122,33864,3455.325,3462.487,I believe that... I want to say one more thing...
6,23122,33864,3462.487,3477.682,"Now, the cases that have described, that you l..."
7,23122,33864,3477.682,3505.865,"In Martin v. OSHRC, Occupational Safety & Heal..."
8,23122,33864,3505.865,3518.204,But in the context of the customs service ruli...
9,23122,15141,3518.204,3519.343,"Thank you, Mr. Jones."


To-do:
 
Table of speech  
Table of justices  
Table of advocates  

# Ad-hoc

In [30]:
case = requests.get('https://api.oyez.org/cases/2019/18-1171')
case = case.json()

In [34]:
case.keys()

dict_keys(['ID', 'name', 'href', 'view_count', 'docket_number', 'additional_docket_numbers', 'manner_of_jurisdiction', 'first_party', 'second_party', 'timeline', 'lower_court', 'facts_of_the_case', 'question', 'conclusion', 'advocates', 'oral_argument_audio', 'citation', 'decisions', 'first_party_label', 'second_party_label', 'heard_by', 'decided_by', 'term', 'location', 'opinion_announcement', 'description', 'written_opinion', 'related_cases', 'justia_url', 'argument2_url'])

In [43]:
oa_df = pd.json_normalize(case['oral_argument_audio'])[['id','title','href']].set_index('id')
oa_df['date'] = oa_df['title'].apply(lambda L:L.split(' - ')[1])
oa_df = oa_df[['date','href']]
oa_df

,date,href
id,,
25088,"November 13, 2019",https://api.oyez.org/case_media/oral_argument_...


In [22]:
advocates = pd.json_normalize(case['advocates'])
advocates

,advocate_description,href,advocate.ID,advocate.name,advocate.href,advocate.view_count,advocate.last_name,advocate.roles,advocate.thumbnail,advocate.length_of_service,advocate.identifier
0,for the petitioner,https://api.oyez.org/case_advocate/case_advoca...,35754,Miguel A. Estrada,https://api.oyez.org/people/miguel_a_estrada,0,Estrada,None,None,0,miguel_a_estrada
1,"Assistant to the Solicitor General, Department...",https://api.oyez.org/case_advocate/case_advoca...,63108,Morgan L. Ratner,https://api.oyez.org/people/morgan_l_ratner,0,Ratner,None,None,0,morgan_l_ratner
2,for the respondents,https://api.oyez.org/case_advocate/case_advoca...,22098,Erwin Chemerinsky,https://api.oyez.org/people/erwin_chemerinsky,0,Chemerinsky,None,None,0,erwin_chemerinsky


In [27]:
#for index in range(len(case['oral_argument_audio'])):
#    print(case(['oral_argument_audio'])[index])
    
oa_1 = requests.get('https://api.oyez.org/case_media/oral_argument_audio/25088')

oa_1 = oa_1.json()

oa_1_df = pd.json_normalize(oa_1)
oa_1_df['date'] = oa_1_df['title'].apply(lambda L:L.split(' - ')[1])
oa_1_df['href'] = 'https://api.oyez.org/case_media/oral_argument_audio/25088'
oa_1_df[['id','date','href']]

,id,date,href
0,25088,"November 13, 2019",https://api.oyez.org/case_media/oral_argument_...


In [ ]:
oa_1

In [9]:
oa_1_transcript = oa_1['transcript']
sections = oa_1_transcript['sections']
petitioner = sections[0]
respondent = sections[1]
petitioner_rebuttal = sections[2]

In [10]:
oa_1_transcript

{'title': 'COMCAST CORPORATION, Petitioner, v. NATIONAL ASSOCIATION OF AFRICAN AMERICAN-OWNED MEDIA, ET AL., Respondents',
 'duration': None,
 'sections': [{'start': 0.195,
   'stop': 1072.91,
   'byte_start': 0,
   'byte_stop': 0,
   'turns': [{'start': 0.195,
     'stop': 11.085,
     'byte_start': 0,
     'byte_stop': 0,
     'speaker': {'ID': 15086,
      'name': 'John G. Roberts, Jr.',
      'href': 'https://api.oyez.org/people/john_g_roberts_jr',
      'view_count': 0,
      'last_name': 'Roberts',
      'roles': [{'id': 2730,
        'type': 'scotus_justice',
        'date_start': 1127970000,
        'date_end': 0,
        'appointing_president': 'George W. Bush',
        'role_title': 'Chief Justice of the United States',
        'institution_name': 'Supreme Court of the United States',
        'href': 'https://api.oyez.org/preson_role/scotus_justice/2730'}],
      'thumbnail': {'id': 32683,
       'mime': 'image/png',
       'size': 52305,
       'href': 'https://api.oyez.org/

In [11]:
section_1_turn_1 = petitioner['turns'][0]
speaker_name = section_1_turn_1['speaker']['name']
text_blocks = section_1_turn_1['text_blocks']
text_block_1 = text_blocks[0]

n_words_spoken = len(text_block_1['text'])

text = text_block_1['text']

n_times_address_counsel = text.count('Counsel') + text.count('counsel')

print(speaker_name,':', n_words_spoken, 'words spoken.', n_times_address_counsel, 'times addressing counsel.')

John G. Roberts, Jr. : 153 words spoken. 0 times addressing counsel.
